# 第 9 章习题与解答

> 本章习题围绕 **answer-only loss masking** 和 **SFT vs Pretrain 的差异**展开。

## Exercise 9.1（易）

**题目**:给定以下 3 轮对话,手动标注哪些 token 的 label ≠ -100(参与 loss)。

```
<bos><|im_start|>user\n你好<|im_end|>\n<bos><|im_start|>assistant\n你好!<|im_end|>\n<bos><|im_start|>user\n再见<|im_end|>\n<bos><|im_start|>assistant\n再见!<|im_end|>
```

<details><summary><b>参考答案</b></summary>

只有两个 **assistant span** 内的 token label ≠ -100:

| 范围 | token | label |
|---|---|---|
| user 「你好」 | 全部 | -100 |
| `<bos><\|im_start\|>assistant\n` | 全部 | -100(标记本身被跳过) |
| assistant 「你好!」 | 你、好、!、`<\|im_end\|>` | **token_id(参与 loss)** |
| `<eos>\n` | eos、\n | **token_id(参与 loss)** |
| user 「再见」 | 全部 | -100 |
| `<bos><\|im_start\|>assistant\n` | 全部 | -100 |
| assistant 「再见!」 | 再、见、!、`<\|im_end\|>` | **token_id(参与 loss)** |
| `<eos>\n` | eos、\n | **token_id(参与 loss)** |

用代码验证:

In [ ]:
import sys
sys.path.insert(0, '/home/minimind')
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('/home/minimind/model')

messages = [
    {"role": "user", "content": "你好"},
    {"role": "assistant", "content": "你好!"},
    {"role": "user", "content": "再见"},
    {"role": "assistant", "content": "再见!"},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
input_ids = tokenizer(prompt, add_special_tokens=False).input_ids

bos_id = tokenizer(f'{tokenizer.bos_token}assistant\n', add_special_tokens=False).input_ids
eos_id = tokenizer(f'{tokenizer.eos_token}\n', add_special_tokens=False).input_ids

labels = [-100] * len(input_ids)
i = 0
while i < len(input_ids):
    if input_ids[i:i + len(bos_id)] == bos_id:
        start = i + len(bos_id)
        end = start
        while end < len(input_ids):
            if input_ids[end:end + len(eos_id)] == eos_id:
                break
            end += 1
        for j in range(start, min(end + len(eos_id), len(input_ids))):
            labels[j] = input_ids[j]
        i = end + len(eos_id)
    else:
        i += 1

n_active = sum(1 for l in labels if l != -100)
n_masked = sum(1 for l in labels if l == -100)
print(f"总 token: {len(input_ids)}, 参与 loss: {n_active}, masked: {n_masked}")
print(f"两个 assistant span 的 token 全部 label≠-100 ✓")

</details>

## Exercise 9.2（中）

**题目**:如果不做 loss masking(把 user 的提问也当训练目标),会出什么问题?

<details><summary><b>参考答案</b></summary>

**三个主要问题:**

1. **模型学会模仿用户提问** —— 训练目标包含 user 的 token,模型会学到「在 assistant 回复之后,应该继续生成 user 的下一个问题」。推理时可能出现自问自答、或者输出完回复后继续输出 `<|im_start|>user\n...` 的情况。

2. **训练信号被稀释** —— 一条对话中,user 部分通常占 30-50% 的 token。如果不 mask,这些 token 也贡献 loss,但它们不是我们想学的。assistant 回复的梯度被「学会提问」的梯度稀释,训练效率下降。

3. **分布偏移** —— 模型在推理时只从 `assistant\n` 之后开始生成,但训练时却学了从 user 部分预测。训练和推理的目标不一致,导致模型困惑。

用代码验证信号稀释:

In [ ]:
import sys
sys.path.insert(0, '/home/minimind')
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('/home/minimind/model')

messages = [
    {"role": "user", "content": "请解释一下什么是机器学习,它在日常生活中有哪些应用?"},
    {"role": "assistant", "content": "机器学习是AI的一个分支。"},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
input_ids = tokenizer(prompt, add_special_tokens=False).input_ids

# 有 masking
bos_id = tokenizer(f'{tokenizer.bos_token}assistant\n', add_special_tokens=False).input_ids
eos_id = tokenizer(f'{tokenizer.eos_token}\n', add_special_tokens=False).input_ids
labels_masked = [-100] * len(input_ids)
i = 0
while i < len(input_ids):
    if input_ids[i:i + len(bos_id)] == bos_id:
        start = i + len(bos_id)
        end = start
        while end < len(input_ids):
            if input_ids[end:end + len(eos_id)] == eos_id:
                break
            end += 1
        for j in range(start, min(end + len(eos_id), len(input_ids))):
            labels_masked[j] = input_ids[j]
        i = end + len(eos_id)
    else:
        i += 1

n_active = sum(1 for l in labels_masked if l != -100)
total = len(input_ids)
print(f"总 token 数:      {total}")
print(f"参与 loss(有 mask): {n_active} ({n_active/total*100:.0f}%)")
print(f"参与 loss(无 mask): {total} (100%)")
print(f"\n→ 无 masking 时,{(1-n_active/total)*100:.0f}% 的梯度花在学「提问」上,浪费了!")

</details>

## Exercise 9.3（难）

**题目**:为什么 SFT 的学习率(1e-5)比预训练(5e-4)低 50 倍?用 catastrophic forgetting 的概念解释。如果用 5e-4 做 SFT 会怎样?

<details><summary><b>参考答案</b></summary>

### catastrophic forgetting(灾难性遗忘)

神经网络在学习新任务时,如果学习率太大,新数据的梯度会大幅更新权重,**覆盖掉之前学到的知识**。这就是 catastrophic forgetting。

### 为什么 SFT 要用低学习率

预训练阶段,模型从零开始,需要大学习率(5e-4)快速学习语言的统计规律(语法、词汇、世界知识)。经过预训练,权重已经到达一个**好的区域**(loss landscape 的低洼处)。

SFT 阶段,模型已经「会语言了」,只需要**微调** —— 让它学会对话格式。如果用 5e-4:
- 梯度太大,权重被大幅拉动,离开预训练找到的好区域
- 模型可能「忘记」预训练学到的知识(语法变乱、常识出错)
- 但对话格式也没学好(训练不充分时数据量太少)

1e-5 足够小,能温和地调整权重,在保留预训练知识的同时学会对话格式。

### 用代码模拟验证

In [ ]:
import torch
import torch.nn as nn

# 模拟:一个已经训练好的简单模型(记住了一个特定 pattern)
torch.manual_seed(42)

model = nn.Linear(10, 1)
# 假装预训练完成:模型学会了 y = sum(x)
with torch.no_grad():
    model.weight.fill_(1.0)
    model.bias.fill_(0.0)

test_input = torch.ones(10)
original_output = model(test_input).item()
print(f"预训练后的输出: {original_output:.4f}  (期望: 10.0)")

# 模拟 SFT:用不同学习率微调
for lr_label, lr in [("1e-5 (SFT)", 1e-5), ("5e-4 (预训练级别)", 5e-4)]:
    m = nn.Linear(10, 1)
    with torch.no_grad():
        m.weight.copy_(torch.ones(10))
        m.bias.copy_(torch.zeros(1))
    
    optimizer = torch.optim.SGD(m.parameters(), lr=lr)
    
    # SFT 数据:让模型输出 5.0(新任务)
    for _ in range(100):
        pred = m(torch.ones(10))
        loss = (pred - 5.0) ** 2
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    final_output = m(test_input).item()
    weight_norm = m.weight.norm().item()
    print(f"\n学习率 {lr_label}:")
    print(f"  SFT 后输出:     {final_output:.4f}  (目标: 5.0)")
    print(f"  权重 L2 范数:    {weight_norm:.4f}")
    print(f"  权重偏离原始(全1): {(m.weight - 1.0).abs().mean().item():.6f}")

print("\n→ 5e-4 的权重偏离更大(catastrophic forgetting 风险更高)")
print("→ 1e-5 温和调整,既学会了新任务,又接近原始权重分布")

**结论**:SFT 的学习率必须是预训练的 1/50(1e-5 vs 5e-4),否则会触发 catastrophic forgetting —— 模型学会对话格式的同时,忘记预训练学到的语言知识。

> 业界经验:SFT 学习率通常是预训练的 **1/10 到 1/100**。minimind 选 1/50 是一个保守但安全的值。更激进的方案(如 LoRA)通过只训练少量参数来避免这个问题,第 11 章会讲。

</details>